In [2]:
import pickle
import mlflow
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [3]:
RUN_ID = 'f12de8218f6d4711a2902c7d4581aac2'

# if server is down, directly point to the location locally or s3 etc.
logged_model = f'/home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/mlartifacts/496409895171607791/{RUN_ID}/artifacts/model'
# logged_model = f'runs:/{RUN_ID}/model' # if mlflow tracking server is available
model = mlflow.pyfunc.load_model(logged_model)

In [4]:
model

mlflow.pyfunc.loaded_model:
  artifact_path: model
  flavor: mlflow.sklearn
  run_id: f12de8218f6d4711a2902c7d4581aac2

In [5]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    return df


def prepare_dictionaries(df: pd.DataFrame):
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [6]:
df = read_dataframe('../web-service-mlflow/data/green_tripdata_2021-01.parquet')

dicts = prepare_dictionaries(df)

y_pred = model.predict(dicts)

In [7]:
y_pred

array([ 6.86271117, 13.36872083,  6.3608707 , ..., 14.43650924,
       37.09262214, 11.10083955])

In [8]:
# let's add the results to dataframe
df_results = pd.DataFrame()

In [9]:
# we don't have ride_id for each ride in dataframe
df.head(5)

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration,PU_DO
0,2,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1.0,43,151,1.0,1.01,5.5,...,0.00,0.0,None,0.3,6.80,2.0,1.0,0.00,3.933333,43_151
1,2,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1.0,166,239,1.0,2.53,10.0,...,2.81,0.0,None,0.3,16.86,1.0,1.0,2.75,8.750000,166_239
2,2,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1.0,41,42,1.0,1.12,6.0,...,1.00,0.0,None,0.3,8.30,1.0,1.0,0.00,5.966667,41_42
3,2,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1.0,168,75,1.0,1.99,8.0,...,0.00,0.0,None,0.3,9.30,2.0,1.0,0.00,7.083333,168_75
7,2,2021-01-01 00:26:31,2021-01-01 00:28:50,N,1.0,75,75,6.0,0.45,3.5,...,0.96,0.0,None,0.3,5.76,1.0,1.0,0.00,2.316667,75_75


In [10]:
## defining the unique id for each row
import uuid

str(uuid.uuid4())

'6f472d93-f9f7-4b2a-928d-15edc05de043'

In [11]:
# defining the same amout of uuid as the number of rows in data

n = len(df)
ride_ids = []
for i in range(n):
    ride_ids.append(str(uuid.uuid4()))

In [12]:
ride_ids[:10]

['6ab56137-9165-45f1-91cf-a75d31c00141',
 '458dfba9-a1f3-4160-b552-a4b7820e5fa2',
 '41124ed7-bb19-44d7-9321-4132b16fe453',
 'f993710a-1472-42dd-a6a2-b692a681d3de',
 'ca694fc6-118f-4703-bfc8-5dcfaaa46a05',
 '7ad94a49-4b10-4868-958a-51df6179514e',
 'd5355504-7352-459f-8a13-0fc23573441a',
 'fe748c08-135d-441a-9ec0-60c3aa4809d9',
 '64206888-5172-4cae-958a-9b94b16af4fd',
 '5de26e01-0f1b-4b7c-af3c-65881e8e528c']

In [13]:
df['ride_id'] = ride_ids

In [14]:
df.head(5)

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration,PU_DO,ride_id
0,2,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1.0,43,151,1.0,1.01,5.5,...,0.0,None,0.3,6.80,2.0,1.0,0.00,3.933333,43_151,6ab56137-9165-45f1-91cf-a75d31c00141
1,2,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1.0,166,239,1.0,2.53,10.0,...,0.0,None,0.3,16.86,1.0,1.0,2.75,8.750000,166_239,458dfba9-a1f3-4160-b552-a4b7820e5fa2
2,2,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1.0,41,42,1.0,1.12,6.0,...,0.0,None,0.3,8.30,1.0,1.0,0.00,5.966667,41_42,41124ed7-bb19-44d7-9321-4132b16fe453
3,2,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1.0,168,75,1.0,1.99,8.0,...,0.0,None,0.3,9.30,2.0,1.0,0.00,7.083333,168_75,f993710a-1472-42dd-a6a2-b692a681d3de
7,2,2021-01-01 00:26:31,2021-01-01 00:28:50,N,1.0,75,75,6.0,0.45,3.5,...,0.0,None,0.3,5.76,1.0,1.0,0.00,2.316667,75_75,ca694fc6-118f-4703-bfc8-5dcfaaa46a05


In [18]:
# let's add the ride_ids to the df_results

df_results['ride_id'] = df['ride_id']
df_results['lpep_pickup_datetime'] = df['lpep_pickup_datetime']
df_results['PULocationID'] = df['PULocationID']
df_results['DOLocationID'] = df['DOLocationID']
df_results['actual_duration'] = df['duration']
df_results['predicted_duration'] = y_pred
df_results['diff'] = df_results['actual_duration'] - df_results['predicted_duration']

In [19]:
df_results

,ride_id,lpep_pickup_datetime,PULocationID,DOLocationID,actual_duration,predicted_duration,diff
0,6ab56137-9165-45f1-91cf-a75d31c00141,2021-01-01 00:15:56,43,151,3.933333,6.862711,-2.929378
1,458dfba9-a1f3-4160-b552-a4b7820e5fa2,2021-01-01 00:25:59,166,239,8.750000,13.368721,-4.618721
2,41124ed7-bb19-44d7-9321-4132b16fe453,2021-01-01 00:45:57,41,42,5.966667,6.360871,-0.394204
3,f993710a-1472-42dd-a6a2-b692a681d3de,2020-12-31 23:57:51,168,75,7.083333,11.824423,-4.741089
7,ca694fc6-118f-4703-bfc8-5dcfaaa46a05,2021-01-01 00:26:31,75,75,2.316667,3.389290,-1.072623
...,...,...,...,...,...,...,...
76513,453552c5-375b-41a8-91de-ee140f38c1fe,2021-01-31 21:38:00,81,90,38.000000,41.526829,-3.526829
76514,fea0ffe4-a568-424d-8d7a-b8a9ecd424af,2021-01-31 22:43:00,35,213,38.000000,43.858974,-5.858974
76515,137cce47-8a48-454b-a983-3f68163f8c72,2021-01-31 22:16:00,74,69,11.000000,14.436509,-3.436509
76516,8798ac03-086e-4913-b122-e8ad597a9ded,2021-01-31 23:10:00,168,215,27.000000,37.092622,-10.092622


In [20]:
!mkdir output

In [26]:
!mkdir output/green

In [22]:
df_results.to_parquet('./output/green_tripdata_2021-01.parquet')

In [23]:
# we can also use like this in start of the code
year = 2021
month = 3
taxi_type = 'green'

input_file = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{taxi_type}_tripdata_{year:04d}-{month:02d}.parquet"
output_file = f'./output/{taxi_type}_tripdata_{year:04d}-{month:02d}.parquet'

In [24]:
input_file

'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-03.parquet'

In [25]:
output_file

'./output/green_tripdata_2021-03.parquet'